In [5]:
# h3_resilience_run.py
# H3: Resilience asymmetry (RPE) — targeted (top-DEB-like) vs random removals.

import os
import math
import pickle
from pathlib import Path
from typing import Dict, List, Tuple, Iterable

import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

ULMM_DIR = Path("ulmm_pickles")
OUT_DIR  = Path("outputs_h3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------
# Helpers
# ---------------------------

def slugify(name: str) -> str:
    import re
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")

def list_ulmm_pickles(directory: Path) -> List[Path]:
    return sorted(directory.glob("ulmm_*.pkl"))

def choose_min_edge_key(G: nx.MultiDiGraph, u: int, v: int, weight_attr: str):
    """
    Return the key of the (u,v,*) edge with minimum weight_attr. If missing, fall back to 'travel_time' then no weight.
    """
    best_key = None
    best_w = float("inf")
    if not G.has_edge(u, v):
        return None
    for k, d in G[u][v].items():
        if weight_attr in d and np.isfinite(d[weight_attr]):
            w = float(d[weight_attr])
        elif "travel_time" in d and np.isfinite(d["travel_time"]):
            w = float(d["travel_time"])
        elif "c_base" in d and np.isfinite(d["c_base"]):
            w = float(d["c_base"])
        else:
            continue
        if w < best_w:
            best_w = w
            best_key = k
    # if no weighted edge found, just take any key
    if best_key is None:
        for k in G[u][v].keys():
            return k
    return best_key

def eligible_edges(G: nx.MultiDiGraph, weight_attr: str) -> List[Tuple[int,int,int]]:
    elig = []
    for u, v, k, d in G.edges(keys=True, data=True):
        w = d.get(weight_attr, d.get("travel_time", d.get("c_base", None)))
        if w is not None and np.isfinite(w):
            elig.append((u, v, k))
    return elig

def dijkstra_tree_multi_source_pred_and_dist(
    G: nx.MultiDiGraph, sources: Iterable[int], weight: str
) -> Tuple[Dict[int, List[int]], Dict[int, float]]:
    """
    Compute a shortest-path tree from a virtual super-source connected to all `sources` by zero-cost links.
    Returns (pred, dist) for every reachable node.
    """
    H = nx.DiGraph()
    # Add all nodes (as a lightweight view)
    H.add_nodes_from(G.nodes())
    # Add weighted arcs: for MultiDiGraph, use min weight among parallel edges
    for u, v, data in G.edges(data=True):
        w = data.get(weight, data.get("travel_time", data.get("c_base", None)))
        if w is None or not np.isfinite(w):
            continue
        # Keep the minimum weight per (u,v)
        if H.has_edge(u, v):
            if w < H[u][v]["weight"]:
                H[u][v]["weight"] = float(w)
        else:
            H.add_edge(u, v, weight=float(w))
    super_src = "__super__"
    H.add_node(super_src)
    for s in sources:
        if s in H:
            H.add_edge(super_src, s, weight=0.0)
    pred, dist = nx.dijkstra_predecessor_and_distance(H, super_src, weight="weight")
    return pred, dist

def reconstruct_path_nodes(pred: Dict[int, List[int]], target: int, stop_node="__super__") -> List[int]:
    """
    Reconstruct a single predecessor path from stop_node to `target`, using the first predecessor if multiple.
    Returns the node list in the direction from source(s) -> target (i.e., super -> ... -> target).
    """
    if target not in pred:
        return []
    path = [target]
    cur = target
    guard = 0
    while True:
        if cur not in pred or len(pred[cur]) == 0:
            # unreachable from super-source
            return []
        p = pred[cur][0]
        if p == stop_node:
            path.reverse()
            return path
        path.append(p)
        cur = p
        guard += 1
        if guard > 2_000_000:  # cycle guard
            return []

def compute_min_time_to_nearest_access(
    G: nx.MultiDiGraph,
    access_nodes: List[int],
    demand_nodes: List[int],
    orientation: str,  # "DA" or "AD"
    weight_attr: str,
) -> np.ndarray:
    """
    Minimal time from each demand to its nearest access, consistent with orientation.
    - DA: d -> a  (compute on G_rev with sources=access, distance to demand)
    - AD: a -> d  (compute on G with sources=access,   distance to demand)
    """
    if orientation not in {"DA", "AD"}:
        raise ValueError("orientation must be 'DA' or 'AD'")
    if orientation == "DA":
        Guse = G.reverse(copy=False)
    else:
        Guse = G
    pred, dist = dijkstra_tree_multi_source_pred_and_dist(Guse, access_nodes, weight=weight_attr)
    out = np.full(len(demand_nodes), np.inf, dtype=float)
    for j, d_node in enumerate(demand_nodes):
        if d_node in dist:
            out[j] = float(dist[d_node])
    return out

def compute_deb_like_edge_scores(
    G: nx.MultiDiGraph,
    access_nodes: List[int],
    demand_nodes: List[int],
    w_d: np.ndarray,
    orientation: str,
    weight_attr: str,
) -> Tuple[Dict[Tuple[int,int,int], float], np.ndarray]:
    """
    Lightweight DEB-like: assign each demand to its nearest access (shortest path consistent with orientation),
    then accumulate w_d along the corresponding path edges. Also returns minimal times per demand (for baseline).
    """
    if orientation == "DA":
        Guse = G.reverse(copy=False)
        map_to_original = True
    else:
        Guse = G
        map_to_original = False

    pred, dist = dijkstra_tree_multi_source_pred_and_dist(Guse, access_nodes, weight=weight_attr)
    edge_score: Dict[Tuple[int,int,int], float] = {}
    base_tau = np.full(len(demand_nodes), np.inf, dtype=float)

    for j, d_node in enumerate(demand_nodes):
        if d_node not in pred or d_node not in dist:
            continue
        base_tau[j] = float(dist[d_node])
        # Reconstruct path nodes in Guse
        path_nodes = reconstruct_path_nodes(pred, d_node, stop_node="__super__")
        if len(path_nodes) < 2:
            continue
        # Convert node path to (u,v,key) edges in ORIGINAL orientation
        for uu, vv in zip(path_nodes[:-1], path_nodes[1:]):
            if map_to_original:
                u, v = vv, uu  # edge in G_rev corresponds to (v->u) in G
            else:
                u, v = uu, vv
            k = choose_min_edge_key(G, u, v, weight_attr)
            if k is None:
                continue
            edge_score[(u, v, k)] = edge_score.get((u, v, k), 0.0) + float(w_d[j])

    return edge_score, base_tau

def compute_rpe_for_demands(
    G: nx.MultiDiGraph,
    access_nodes: List[int],
    demand_nodes: List[int],
    orientation: str,  # "DA" or "AD"
    weight_attr: str,
    K: int = 10,
    eta: float = 1.0,
) -> np.ndarray:
    """
    Oriented RPE per demand using K nearest access shortest-path lengths and a Gibbs distribution.
    DA: distances on G   from d -> a
    AD: distances on G^R from d -> a  (equivalently a -> d on G)
    """
    if orientation not in {"DA", "AD"}:
        raise ValueError("orientation must be 'DA' or 'AD'")
    if orientation == "DA":
        Guse = G
    else:
        Guse = G.reverse(copy=False)

    rpe = np.zeros(len(demand_nodes), dtype=float)
    for j, d_node in enumerate(demand_nodes):
        try:
            dist_map = nx.single_source_dijkstra_path_length(Guse, d_node, weight=weight_attr)
        except Exception:
            dist_map = {}
        # collect distances to access nodes
        vals = [dist_map[a] for a in access_nodes if a in dist_map]
        if not vals:
            rpe[j] = 0.0
            continue
        vals = np.array(vals, dtype=float)
        vals = np.sort(vals)[:min(K, len(vals))]
        # Gibbs weights
        w = np.exp(-eta * vals)
        if w.sum() <= 0:
            rpe[j] = 0.0
            continue
        p = w / w.sum()
        rpe[j] = float(-(p * np.log(p + 1e-12)).sum())
    return rpe

def remove_edges_copy(G: nx.MultiDiGraph, edges_to_remove: List[Tuple[int,int,int]]) -> nx.MultiDiGraph:
    H = G.copy()
    for (u, v, k) in edges_to_remove:
        if H.has_edge(u, v, key=k):
            H.remove_edge(u, v, key=k)
    return H

def tertile_bins(x: np.ndarray) -> np.ndarray:
    """
    Return tertile labels {0,1,2} for array x, breaking ties by rank.
    """
    x = np.asarray(x, float)
    mask = np.isfinite(x)
    ranks = np.full_like(x, np.nan, dtype=float)
    ranks[mask] = (x[mask].argsort().argsort() + 1) / (mask.sum() + 1e-12)
    # tertiles by rank
    out = np.full_like(x, -1, dtype=int)
    out[(ranks >= 0.0) & (ranks < 1/3)] = 0
    out[(ranks >= 1/3) & (ranks < 2/3)] = 1
    out[(ranks >= 2/3)] = 2
    return out

def summarize_deltas_by_tertile(
    deltas: np.ndarray, tertiles: np.ndarray
) -> pd.DataFrame:
    """
    Returns DataFrame with {tertile, median, iqr, n, frac_inf}
    """
    rows = []
    for t in [0, 1, 2]:
        sel = (tertiles == t) & np.isfinite(deltas)
        vals = deltas[sel]
        if vals.size == 0:
            med = np.nan; iqr = np.nan; n = 0
        else:
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
            iqr = q3 - q1
            n = vals.size
        frac_inf = float(np.mean(~np.isfinite(deltas[tertiles == t]))) if np.any(tertiles == t) else 0.0
        rows.append(dict(tertile=t, median=med, iqr=iqr, n=n, frac_inf=frac_inf))
    return pd.DataFrame(rows)

# ---------------------------
# Main H3 runner for one city
# ---------------------------

def run_h3_for_city(
    pkl_path: Path,
    out_dir: Path,
    vehicles: List[str] = None,
    orientations: List[str] = ("DA", "AD"),
    weight_key_template: str = "c_eff_{veh}",
    budgets_pct: List[float] = (2.0,),    # remove top p% edges
    random_reps: int = 10,
    K_rpe: int = 10,
    eta_rpe: float = 1.0,
    rng_seed: int = 42,
) -> pd.DataFrame:
    with open(pkl_path, "rb") as f:
        ulmm = pickle.load(f)

    city = ulmm.get("city", slugify(pkl_path.stem))
    city_slug = slugify(city)
    G: nx.MultiDiGraph = ulmm["graph"]
    dfD: pd.DataFrame = ulmm["demand"]
    dfA: pd.DataFrame = ulmm["access"]
    params: Dict = ulmm.get("params", {})

    if dfA.empty or dfD.empty:
        print(f"[SKIP] {city}: access={len(dfA)} demand={len(dfD)}")
        return pd.DataFrame()

    if vehicles is None:
        veh_list = list(params.get("vehicles", {"van": {}, "cargo_bike": {}}).keys())
    else:
        veh_list = vehicles

    A_nodes = dfA["i_node"].astype(int).tolist()
    D_nodes = dfD["i_node"].astype(int).tolist()
    w_d = dfD["w"].astype(float).to_numpy()

    city_out = out_dir / city_slug
    city_out.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(rng_seed)
    summaries = []

    for veh in veh_list:
        weight_attr = weight_key_template.format(veh=veh)
        # quick check: at least one edge has this attribute
        if not any(weight_attr in d for _,_,_,d in G.edges(keys=True, data=True)):
            print(f"[WARN] {city} [{veh}]: '{weight_attr}' not on edges. Skipping.")
            continue

        for orient in orientations:
            print(f"\n=== H3 {city} | {veh} | {orient} ===")

            # --- RPE tertiles on baseline graph
            rpe = compute_rpe_for_demands(G, A_nodes, D_nodes, orientation=orient, weight_attr=weight_attr,
                                          K=K_rpe, eta=eta_rpe)
            tert = tertile_bins(rpe)

            # --- Edge scores (DEB-like) + baseline tau_min per demand
            edge_score, base_tau = compute_deb_like_edge_scores(
                G, A_nodes, D_nodes, w_d, orientation=orient, weight_attr=weight_attr
            )

            # Keep list of all edge keys (universe for random)
            all_edge_keys = eligible_edges(G, weight_attr)

            # Baseline minimal times (recomputed robustly; matches base_tau)
            tau0 = compute_min_time_to_nearest_access(G, A_nodes, D_nodes, orient, weight_attr)

            # Safe denominator
            eps = 1e-9
            valid_mask = np.isfinite(tau0)
            tau0 = np.where(valid_mask, tau0, np.nan)

            # --- For each budget p%
            for p in budgets_pct:
                n_target = max(1, int(round(len(all_edge_keys) * (p / 100.0))))
                ranked = sorted(edge_score.items(), key=lambda kv: kv[1], reverse=True)
                n_eff = min(n_target, len(ranked))  # stesso numero per T e R
                targeted_edges = [ek for ek, _ in ranked[:n_eff]]

                # Remove targeted and recompute tau'
                G_targeted = remove_edges_copy(G, targeted_edges)
                tauT = compute_min_time_to_nearest_access(G_targeted, A_nodes, D_nodes, orient, weight_attr)
                # Relative delta per demand (skip infinities in summary but track)
                deltaT = (tauT - tau0) / (tau0 + eps)
                deltaT[~valid_mask] = np.nan

                # Random replicates
                all_rand_deltas = []
                all_rand_tau    = []
                for r in range(random_reps):
                    rand_idx = rng.choice(len(all_edge_keys), size=n_eff, replace=False)
                    edges_r = [all_edge_keys[idx] for idx in rand_idx]
                    G_rand = remove_edges_copy(G, edges_r)
                    tauR = compute_min_time_to_nearest_access(G_rand, A_nodes, D_nodes, orient, weight_attr)
                    all_rand_tau.append(tauR)
                
                    deltaR = (tauR - tau0) / (tau0 + eps)
                    deltaR[~valid_mask] = np.nan
                    all_rand_deltas.append(deltaR)

                # --- Pool random reps (shape: reps x |D|)
                deltasR_pool = np.vstack(all_rand_deltas)
                
                # Columns (demands) that have at least one finite value across reps
                cols_ok = np.isfinite(deltasR_pool).any(axis=0)
                
                # Column-wise nanmedian per ok columns (no warnings), NaN otherwise
                rand_med = np.full(deltasR_pool.shape[1], np.nan, dtype=float)
                if cols_ok.any():
                    rand_med[cols_ok] = np.nanmedian(deltasR_pool[:, cols_ok], axis=0)
                
                # --- Summaries by tertile (mediane condizionali alla raggiungibilità)
                dfT = summarize_deltas_by_tertile(deltaT, tert)
                dfR = summarize_deltas_by_tertile(rand_med, tert)
                
                # Extra diagnostic: share of demands with no finite random delta in each tertile
                nofinite_share = []
                for t in [0, 1, 2]:
                    tmask = (tert == t)
                    if not np.any(tmask):
                        nofinite_share.append(np.nan)
                    else:
                        nofinite_share.append(float(np.mean(~cols_ok[tmask])))
                dfR["share_all_nan"] = nofinite_share
                
                # --- NEW: isolamento vero dopo lo shock (baseline finita -> post infinito)
                # Targeted isolation (boolean per domanda)
                isoT = valid_mask & (~np.isfinite(tauT))
                
                # Random isolation: frazione di repliche in cui il demand si isola
                isoR_mat = np.vstack([(valid_mask & (~np.isfinite(tr))).astype(float) for tr in all_rand_tau])
                isoR_rate = isoR_mat.mean(axis=0)  # ∈ [0,1] per domanda
                
                def tertile_rate(vec_bool_or_rate, tertiles, valid0):
                    """Media per terzile su nodi baseline-validi (Low=0,Mid=1,High=2)."""
                    out = []
                    for t in [0, 1, 2]:
                        tm = (tertiles == t) & valid0
                        out.append(float(np.mean(vec_bool_or_rate[tm])) if np.any(tm) else np.nan)
                    return out
                
                # % isolamento per terzile (T: booleano; R: soglia ‘maggioranza repliche’)
                isoT_by_t = tertile_rate(isoT.astype(float), tert, valid_mask)
                isoR_by_t = tertile_rate((isoR_rate > 0.5).astype(float), tert, valid_mask)
                # Se preferisci riportare la media di isoR_rate (quota attesa di repliche),
                # usa: isoR_by_t = tertile_rate(isoR_rate, tert, valid_mask)
                
                # --- Mann-Whitney su terzile Low (T vs pooled-R), guarding against empties
                low_mask = (tert == 0) & np.isfinite(deltaT)
                targ_low_vals = deltaT[low_mask]
                
                rand_low_chunks = []
                for arr in all_rand_deltas:
                    chunk = arr[low_mask]
                    if np.any(np.isfinite(chunk)):
                        rand_low_chunks.append(chunk[np.isfinite(chunk)])
                
                if targ_low_vals.size > 0 and len(rand_low_chunks) > 0:
                    rand_low_vals = np.concatenate(rand_low_chunks)
                    if rand_low_vals.size > 0:
                        try:
                            _, pval_low = mannwhitneyu(targ_low_vals, rand_low_vals, alternative="greater")
                        except Exception:
                            pval_low = np.nan
                    else:
                        pval_low = np.nan
                else:
                    pval_low = np.nan
                
                # --- Collect compact row (aggiunte le colonne di isolamento)
                row = dict(
                    City=city,
                    Vehicle=veh,
                    Orientation=orient,
                    BudgetPct=p,
                    Low_T_median=dfT.loc[dfT.tertile==0, "median"].values[0],
                    Low_R_median=dfR.loc[dfR.tertile==0, "median"].values[0],
                    Mid_T_median=dfT.loc[dfT.tertile==1, "median"].values[0],
                    Mid_R_median=dfR.loc[dfR.tertile==1, "median"].values[0],
                    High_T_median=dfT.loc[dfT.tertile==2, "median"].values[0],
                    High_R_median=dfR.loc[dfR.tertile==2, "median"].values[0],
                    Low_pval=pval_low,
                    # (legacy) frazione di inf nel vettore Δ per terzile — puoi mantenerla
                    Low_frac_inf_T=dfT.loc[dfT.tertile==0, "frac_inf"].values[0],
                    Mid_frac_inf_T=dfT.loc[dfT.tertile==1, "frac_inf"].values[0],
                    High_frac_inf_T=dfT.loc[dfT.tertile==2, "frac_inf"].values[0],
                    # diagnostica random
                    Low_share_all_nan_R=dfR.loc[dfR.tertile==0, "share_all_nan"].values[0],
                    Mid_share_all_nan_R=dfR.loc[dfR.tertile==1, "share_all_nan"].values[0],
                    High_share_all_nan_R=dfR.loc[dfR.tertile==2, "share_all_nan"].values[0],
                    # NEW: isolamento per terzile (quote ∈ [0,1])
                    Low_iso_T = isoT_by_t[0],
                    Mid_iso_T = isoT_by_t[1],
                    High_iso_T = isoT_by_t[2],
                    Low_iso_R = isoR_by_t[0],
                    Mid_iso_R = isoR_by_t[1],
                    High_iso_R = isoR_by_t[2],
                )
                summaries.append(row)

                # Per-city figure (optional)
                fig, ax = plt.subplots(figsize=(5.2, 3.6), dpi=140)
                tert_names = ["Low RPE", "Mid RPE", "High RPE"]
                x = np.arange(3)
                # Bars: use medians with IQR as errorbars; show targeted and random side-by-side
                T_meds = dfT["median"].to_numpy()
                T_iqr  = dfT["iqr"].to_numpy()
                R_meds = dfR["median"].to_numpy()
                R_iqr  = dfR["iqr"].to_numpy()
                width = 0.35
                ax.bar(x - width/2, T_meds*100, width, yerr=T_iqr*100, capsize=4, label="Targeted (T)")
                ax.bar(x + width/2, R_meds*100, width, yerr=R_iqr*100, capsize=4, label="Random (R)")
                ax.set_xticks(x)
                ax.set_xticklabels(tert_names, rotation=0)
                ax.set_ylabel("Median Δτ (%)")
                ax.set_title(f"{city} – {veh} – {orient} – p={p:.1f}%")
                ax.grid(True, axis="y", alpha=0.25)
                ax.legend(loc="upper left")
                fig.tight_layout()
                fpath = city_out / f"fig-shock-sensitivity_{slugify(veh)}_{orient}_p{int(round(p))}.png"
                fig.savefig(fpath, bbox_inches="tight")
                plt.close(fig)

    # Save per-city summary CSV
    df_sum = pd.DataFrame(summaries)
    if not df_sum.empty:
        df_sum.to_csv(city_out / "h3_summary_city.csv", index=False)
    return df_sum

# ---------------------------
# Pooled run over all cities
# ---------------------------

def run_h3_all(
    ulmm_dir: Path = ULMM_DIR,
    out_dir: Path = OUT_DIR,
    vehicles: List[str] = None,
    orientations: List[str] = ("DA", "AD"),
    budgets_pct: List[float] = (2.0,),
    random_reps: int = 10,
    K_rpe: int = 10,
    eta_rpe: float = 1.0,
    rng_seed: int = 42,
):
    pickles = list_ulmm_pickles(ulmm_dir)
    if not pickles:
        print(f"No ULMM pickles in: {ulmm_dir}")
        return

    all_summaries = []
    for pkl in pickles:
        try:
            city_df = run_h3_for_city(
                pkl, out_dir, vehicles=vehicles, orientations=orientations,
                budgets_pct=budgets_pct, random_reps=random_reps,
                K_rpe=K_rpe, eta_rpe=eta_rpe, rng_seed=rng_seed
            )
            if not city_df.empty:
                all_summaries.append(city_df)
        except Exception as e:
            print(f"[ERROR] {pkl.name}: {e}")

    if not all_summaries:
        print("No H3 results produced.")
        return

    full = pd.concat(all_summaries, ignore_index=True)
    full_csv = out_dir / "h3_summary_ALL.csv"
    full.to_csv(full_csv, index=False)
    print(f"Saved: {full_csv.resolve()}")

    # Compact LaTeX-style table per city (Low/Mid/High medians, T vs R, and p-val)
    # Use the first (or only) budget in budgets_pct
    p = budgets_pct[0]
    rows = []
    for (city, veh, orient), g in full.groupby(["City", "Vehicle", "Orientation"]):
        g = g[g["BudgetPct"] == p]
        if g.empty:
            continue
        # collapse across veh/orient if desired; here we keep them separate for clarity
        rows.append([
            city, veh, orient, p,
            g["Low_T_median"].median()*100, g["Low_R_median"].median()*100, g["Low_pval"].median(),
            g["Mid_T_median"].median()*100, g["Mid_R_median"].median()*100,
            g["High_T_median"].median()*100, g["High_R_median"].median()*100
        ])
    df_tex = pd.DataFrame(rows, columns=[
        "City","Vehicle","Orient","Budget(%)",
        "Low T","Low R","p_low",
        "Mid T","Mid R",
        "High T","High R"
    ])
    df_tex.to_csv(out_dir / "h3_table_compact.csv", index=False)

    # Pooled figure (three tertiles; medians across all city-veh-orient rows)
    pooled = []
    for tname, tsel in [("Low RPE", "Low_T_median"), ("Mid RPE","Mid_T_median"), ("High RPE","High_T_median")]:
        pooled.append(("T", tname, np.nanmedian(full[tsel])*100))
    for tname, tsel in [("Low RPE", "Low_R_median"), ("Mid RPE","Mid_R_median"), ("High RPE","High_R_median")]:
        pooled.append(("R", tname, np.nanmedian(full[tsel])*100))
    plot_df = pd.DataFrame(pooled, columns=["Type","Tertile","MedianPct"])

    fig, ax = plt.subplots(figsize=(6.8, 3.8), dpi=150)
    tertiles = ["Low RPE", "Mid RPE", "High RPE"]
    x = np.arange(3)
    width = 0.35
    T_vals = [plot_df[(plot_df.Type=="T") & (plot_df.Tertile==tn)]["MedianPct"].values[0] for tn in tertiles]
    R_vals = [plot_df[(plot_df.Type=="R") & (plot_df.Tertile==tn)]["MedianPct"].values[0] for tn in tertiles]
    ax.bar(x - width/2, T_vals, width, label="Targeted (T)")
    ax.bar(x + width/2, R_vals, width, label="Random (R)")
    ax.set_xticks(x); ax.set_xticklabels(tertiles)
    ax.set_ylabel("Pooled median Δτ (%)")
    ax.set_title(f"Shock sensitivity by RPE tertile (pooled, p={p:.1f}%)")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(loc="upper left")
    fig.tight_layout()
    fig.savefig(out_dir / "fig-shock-sensitivity.png", bbox_inches="tight")
    plt.close(fig)

run_h3_all(
    ulmm_dir=ULMM_DIR,
    out_dir=OUT_DIR,
    vehicles=None,                     # use vehicles present in each pickle
    orientations=("DA","AD"),
    budgets_pct=[2.0],                 # % of edges removed
    random_reps=10,
    K_rpe=10,
    eta_rpe=1.0,
    rng_seed=42,
)


=== H3 Amsterdam, Netherlands | van | DA ===

=== H3 Amsterdam, Netherlands | van | AD ===

=== H3 Amsterdam, Netherlands | cargo_bike | DA ===

=== H3 Amsterdam, Netherlands | cargo_bike | AD ===

=== H3 Barcelona, Spain | van | DA ===

=== H3 Barcelona, Spain | van | AD ===

=== H3 Barcelona, Spain | cargo_bike | DA ===

=== H3 Barcelona, Spain | cargo_bike | AD ===

=== H3 New York City, New York, USA | van | DA ===

=== H3 New York City, New York, USA | van | AD ===

=== H3 New York City, New York, USA | cargo_bike | DA ===

=== H3 New York City, New York, USA | cargo_bike | AD ===

=== H3 Paris, France | van | DA ===

=== H3 Paris, France | van | AD ===

=== H3 Paris, France | cargo_bike | DA ===

=== H3 Paris, France | cargo_bike | AD ===

=== H3 Seattle, Washington, USA | van | DA ===

=== H3 Seattle, Washington, USA | van | AD ===

=== H3 Seattle, Washington, USA | cargo_bike | DA ===

=== H3 Seattle, Washington, USA | cargo_bike | AD ===
Saved: /Users/ecorradini/Papers/Logist

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out